# 06 — AOI Transfer Test

**Purpose:** Check whether existing land-cover outputs and available reference data cover the study area consistently. This notebook is a *transfer/coverage diagnostic*, not an accuracy claim by itself.

It does not train a model, edit source data, or overwrite project outputs. It looks for available AOI files, classification rasters, and reference/label files, then reports what can be compared.

**Important limitation:** A genuine accuracy assessment needs independent reference labels for the target AOI. Raster overlap, class proportions, or visual similarity alone cannot establish classification accuracy.

## 1. Imports and project paths

Run this notebook from the project root or its `notebooks/` folder. The project root is detected from common project files.

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import transform_bounds
from rasterio.features import geometry_mask

try:
    import geopandas as gpd
except ImportError:
    gpd = None

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (p.resolve() for p in CANDIDATE_ROOTS
     if (p / "data").exists() and (p / "outputs").exists()),
    Path.cwd().resolve()
)

DATA_DIR = PROJECT_ROOT / "data"
AOI_DIR = DATA_DIR / "aoi"
REFERENCE_DIR = DATA_DIR / "reference"
CLASSIFICATION_DIR = PROJECT_ROOT / "outputs" / "classification"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

print("Project root:", PROJECT_ROOT)
print("AOI directory exists:", AOI_DIR.exists())
print("Reference directory exists:", REFERENCE_DIR.exists())
print("Classification directory exists:", CLASSIFICATION_DIR.exists())

Project root: C:\Users\lenovo\Desktop\satellite-land-cover-classification
AOI directory exists: True
Reference directory exists: True
Classification directory exists: True


## 2. Inventory candidate AOI, reference, and classification files

This section only inventories files; it does not assume that every candidate is a valid independent reference source.

In [2]:
def list_files(folder, suffixes):
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in suffixes
    )

aoi_files = list_files(AOI_DIR, {".geojson", ".json", ".gpkg", ".shp"})
reference_files = list_files(REFERENCE_DIR, {".geojson", ".json", ".gpkg", ".shp", ".tif", ".tiff", ".csv"})
classification_files = list_files(CLASSIFICATION_DIR, {".tif", ".tiff"})

print("AOI candidates:")
for p in aoi_files:
    print(" -", p.relative_to(PROJECT_ROOT))
print("\nReference candidates:")
for p in reference_files:
    print(" -", p.relative_to(PROJECT_ROOT))
print("\nClassification rasters:")
for p in classification_files:
    print(" -", p.relative_to(PROJECT_ROOT))

AOI candidates:
 - data\aoi\aoi.gpkg
 - data\aoi\study_area.geojson
 - data\aoi\training_samples.gpkg
 - data\aoi\training_samples_v2.gpkg

Reference candidates:
 - data\reference\DynamicWorld_candidate_samples.csv

Classification rasters:
 - outputs\classification\land_cover_baseline.tif
 - outputs\classification\land_cover_extra_trees.tif


## 3. Inspect AOI vector layers

For each readable vector file, report CRS, feature count, and bounds. If multiple files represent the same AOI, treat them as candidates until their provenance is confirmed.

In [3]:
aoi_summaries = []
aoi_layers = {}

if gpd is None:
    print("GeoPandas is not installed. Install it in this environment to inspect vector AOIs.")
else:
    for path in aoi_files:
        try:
            layer = gpd.read_file(path)
            aoi_layers[str(path)] = layer
            aoi_summaries.append({
                "file": str(path.relative_to(PROJECT_ROOT)),
                "crs": str(layer.crs),
                "features": len(layer),
                "bounds": tuple(layer.total_bounds) if len(layer) else None,
                "geometry_types": ", ".join(sorted(layer.geom_type.dropna().unique()))
            })
        except Exception as exc:
            aoi_summaries.append({
                "file": str(path.relative_to(PROJECT_ROOT)),
                "crs": "READ ERROR",
                "features": None,
                "bounds": str(exc),
                "geometry_types": ""
            })

aoi_summary_df = pd.DataFrame(aoi_summaries)
display(aoi_summary_df)

c:\Users\lenovo\Desktop\satellite-land-cover-classification\.venv\Lib\site-packages\pyogrio\raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(


,file,crs,features,bounds,geometry_types
0,data\aoi\aoi.gpkg,EPSG:4326,1,"(72.55744464459157, 23.155963163960344, 72.608...",Polygon
1,data\aoi\study_area.geojson,EPSG:4326,1,"(72.55744464459157, 23.155963163960344, 72.608...",Polygon
2,data\aoi\training_samples.gpkg,EPSG:4326,39,"(72.55812823013694, 23.156978452640594, 72.605...",Point
3,data\aoi\training_samples_v2.gpkg,EPSG:4326,39,"(72.55812823013694, 23.156978452640594, 72.605...",Point


## 4. Inspect classification raster metadata

The notebook reports each raster's CRS, resolution, dimensions, nodata value, and bounds. This helps identify whether the raster grid can be compared with the selected AOI.

In [4]:
raster_summaries = []
for path in classification_files:
    try:
        with rasterio.open(path) as src:
            raster_summaries.append({
                "file": str(path.relative_to(PROJECT_ROOT)),
                "crs": str(src.crs),
                "width": src.width,
                "height": src.height,
                "resolution": tuple(src.res),
                "nodata": src.nodata,
                "bounds": tuple(src.bounds),
                "valid_data_pixels": int(np.count_nonzero(src.read(1, masked=True).compressed()))
            })
    except Exception as exc:
        raster_summaries.append({
            "file": str(path.relative_to(PROJECT_ROOT)),
            "crs": "READ ERROR",
            "width": None, "height": None, "resolution": None,
            "nodata": None, "bounds": str(exc), "valid_data_pixels": None
        })

raster_summary_df = pd.DataFrame(raster_summaries)
display(raster_summary_df)

,file,crs,width,height,resolution,nodata,bounds,valid_data_pixels
0,outputs\classification\land_cover_baseline.tif,EPSG:32642,528,280,"(10.0, 10.0)",0.0,"(864240.0, 2565250.0, 869520.0, 2568050.0)",138377
1,outputs\classification\land_cover_extra_trees.tif,EPSG:32642,528,280,"(10.0, 10.0)",0.0,"(864240.0, 2565250.0, 869520.0, 2568050.0)",138377


## 5. Compare AOI bounds with classification raster coverage

Choose an AOI layer below. Bounds are transformed to the raster CRS before comparison. A bounds overlap is only a first check; the geometry mask then estimates how many raster pixels fall inside the AOI.

In [5]:
if not aoi_layers:
    print("No readable AOI vector layer found. Check the inventory above and GeoPandas installation.")
else:
    aoi_options = list(aoi_layers.keys())
    print("AOI options:")
    for i, key in enumerate(aoi_options):
        print(f"{i}: {Path(key).relative_to(PROJECT_ROOT)}")

AOI options:
0: data\aoi\aoi.gpkg
1: data\aoi\study_area.geojson
2: data\aoi\training_samples.gpkg
3: data\aoi\training_samples_v2.gpkg


In [6]:
# Set this index to the AOI layer you intend to test.
AOI_CHOICE = 0

if aoi_layers and classification_files:
    selected_aoi_path = aoi_options[AOI_CHOICE]
    selected_aoi = aoi_layers[selected_aoi_path]

    if selected_aoi.crs is None:
        print("Selected AOI has no CRS; cannot compare it safely.")
    elif selected_aoi.empty:
        print("Selected AOI is empty.")
    else:
        geom_union = selected_aoi.geometry.union_all()
        coverage_rows = []

        for raster_path in classification_files:
            with rasterio.open(raster_path) as src:
                aoi_in_raster_crs = selected_aoi.to_crs(src.crs)
                geom = aoi_in_raster_crs.geometry.union_all()
                raster_bounds_geom = gpd.GeoSeries(
                    [gpd.GeoSeries.from_bbox(src.bounds).iloc[0]],
                    crs=src.crs
                ).iloc[0] if False else None

                aoi_bounds = aoi_in_raster_crs.total_bounds
                b = src.bounds
                bounds_overlap = not (
                    aoi_bounds[2] <= b.left or aoi_bounds[0] >= b.right or
                    aoi_bounds[3] <= b.bottom or aoi_bounds[1] >= b.top
                )

                inside = geometry_mask(
                    [geom], out_shape=(src.height, src.width),
                    transform=src.transform, invert=True, all_touched=False
                )
                inside_count = int(inside.sum())
                values = src.read(1, masked=True)
                valid_inside = inside & ~np.ma.getmaskarray(values)
                valid_count = int(valid_inside.sum())

                coverage_rows.append({
                    "aoi": str(Path(selected_aoi_path).relative_to(PROJECT_ROOT)),
                    "raster": str(raster_path.relative_to(PROJECT_ROOT)),
                    "bounds_overlap": bounds_overlap,
                    "AOI_grid_pixels": inside_count,
                    "valid_classified_pixels_inside_AOI": valid_count,
                    "valid_coverage_percent_of_AOI_grid": (
                        100 * valid_count / inside_count if inside_count else np.nan
                    )
                })

        coverage_df = pd.DataFrame(coverage_rows)
        display(coverage_df)
else:
    print("Need at least one readable AOI and one classification raster to run the coverage comparison.")

,aoi,raster,bounds_overlap,AOI_grid_pixels,valid_classified_pixels_inside_AOI,valid_coverage_percent_of_AOI_grid
0,data\aoi\aoi.gpkg,outputs\classification\land_cover_baseline.tif,True,138936,138377,99.597656
1,data\aoi\aoi.gpkg,outputs\classification\land_cover_extra_trees.tif,True,138936,138377,99.597656


## 6. Inspect reference data candidates

A reference file is not automatically independent ground truth. Confirm how labels were collected, their dates, class definitions, and whether they were used to train or tune the model before using them for evaluation.

In [7]:
reference_inventory = []
for path in reference_files:
    row = {
        "file": str(path.relative_to(PROJECT_ROOT)),
        "type": path.suffix.lower(),
        "size_bytes": path.stat().st_size
    }
    try:
        if path.suffix.lower() == ".csv":
            sample = pd.read_csv(path, nrows=5)
            row["columns_or_bands"] = ", ".join(map(str, sample.columns))
            row["sample_rows_read"] = len(sample)
        elif path.suffix.lower() in {".tif", ".tiff"}:
            with rasterio.open(path) as src:
                row["columns_or_bands"] = f"{src.count} band(s), CRS={src.crs}, shape={src.height}x{src.width}"
        elif gpd is not None:
            layer = gpd.read_file(path, rows=5)
            row["columns_or_bands"] = ", ".join(map(str, layer.columns))
    except Exception as exc:
        row["columns_or_bands"] = f"Could not inspect: {exc}"
    reference_inventory.append(row)

display(pd.DataFrame(reference_inventory))

,file,type,size_bytes,columns_or_bands,sample_rows_read
0,data\reference\DynamicWorld_candidate_samples.csv,.csv,40876,"system:index, class_id, .geo",5


## 7. Optional class-distribution comparison

If a reference classification raster exists and uses the **same class codes and definitions**, compare its class counts with a prediction raster over their common valid area. This is a distribution diagnostic, not an accuracy metric.

In [8]:
# Review the file inventory first, then set these paths to compatible rasters.
PREDICTION_RASTER = None  # Example: PROJECT_ROOT / "outputs/classification/land_cover_baseline.tif"
REFERENCE_CLASS_RASTER = None  # Set only if a genuine, compatible reference label raster exists.

def class_counts_on_common_grid(pred_path, ref_path):
    with rasterio.open(pred_path) as pred, rasterio.open(ref_path) as ref:
        if pred.crs != ref.crs or pred.transform != ref.transform or pred.shape != ref.shape:
            raise ValueError(
                "Rasters do not share the same grid. Reproject/resample carefully before comparison; "
                "categorical labels require nearest-neighbour resampling."
            )
        p = pred.read(1, masked=True)
        r = ref.read(1, masked=True)
        valid = ~np.ma.getmaskarray(p) & ~np.ma.getmaskarray(r)
        valid &= np.isfinite(p.filled(np.nan)) & np.isfinite(r.filled(np.nan))
        p_values = p.data[valid].astype(int)
        r_values = r.data[valid].astype(int)
        classes = sorted(set(np.unique(p_values)) | set(np.unique(r_values)))
        rows = []
        for cls in classes:
            rows.append({
                "class_id": cls,
                "prediction_pixels": int((p_values == cls).sum()),
                "reference_pixels": int((r_values == cls).sum())
            })
        return pd.DataFrame(rows), int(valid.sum())

if PREDICTION_RASTER is not None and REFERENCE_CLASS_RASTER is not None:
    counts_df, common_pixels = class_counts_on_common_grid(
        Path(PREDICTION_RASTER), Path(REFERENCE_CLASS_RASTER)
    )
    print("Common valid pixels:", common_pixels)
    display(counts_df)
else:
    print("Skipped: set both raster paths only when a compatible reference class raster is available.")

Skipped: set both raster paths only when a compatible reference class raster is available.


## 8. Conclusions and limitations

Use the output to answer:

1. Does the selected AOI overlap each classification raster?
2. What percentage of the AOI grid has valid classified pixels?
3. Are there reference labels for this target AOI, and are they independent of training/tuning?
4. Are class definitions and class codes consistent across data sources?

**Do not report transfer accuracy** unless you have independent reference labels and a defensible comparison protocol. Coverage, map appearance, and class proportions are useful diagnostics but are not substitutes for validation.